# GVGAI WAN VAE End-to-End Colab Pipeline

Run this notebook on a Colab H100 runtime. It collects GVGAI RGB transitions for `aliens`, `chopper`, and `waves` across stock, multishot, and ricochet variants, encodes transitions with the pretrained WAN VAE, trains the dynamics model, starts TensorBoard, and writes checkpoints plus preview images to Google Drive.

The defaults are a conservative 7-hour end-to-end preset: every requested game/variant is included, but frame counts, training steps, and validation work are capped so a single overnight Colab session has a realistic chance to finish.

In [ ]:
# Colab and experiment configuration
from pathlib import Path

REPO_URL = "https://github.com/wessimpson/DecoupliWo.git"
REPO_BRANCH = "pranit-wan-vae-pretrained"
GVGAI_BRANCH = "gvgai-aliens-train-variants"
GVGAI_COMMIT = "91a74036fc3376d3db6eb1fdf6136c7d6604e886"

REPO_DIR = Path("/content/DecoupliWo")
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/DecoupliWo/gvgai_wan_vae_pretrained")

GAME_STEMS = [
    "aliens",
    "aliens_rules_multishot",
    "aliens_rules_ricochet",
    "chopper",
    "chopper_rules_multishot",
    "chopper_rules_ricochet",
    "waves",
    "waves_rules_multishot",
    "waves_rules_ricochet",
    "defender",
    "jaws",
]

# 7-hour preset: all 9 stems, reduced frame budget, reduced training/validation work.
TRAIN_FRAMES_PER_STEM = 20_000
TEST_FRAMES_PER_STEM = 2_000
COLLECT_NUM_ENVS = 8
COLLECT_CHUNK_SIZE = 5_000
COLLECT_SCALE = 0.5
COLLECT_SEED = 42
FORCE_RECOLLECT = False

ENCODE_BATCH_SIZE = 16
ENCODE_DOWN_SCALE = 1
FORCE_REENCODE = True

TRAIN_BATCH_SIZE = 8
TRAIN_NUM_WORKERS = 4
TRAIN_MAX_STEPS = 20_000
TRAIN_NUM_EPOCHS = 50
TRAIN_LR = 5e-5
TRAIN_SAVE_EVERY = 2_500
TRAIN_VALIDATION_EVERY = 5_000
TRAIN_VAL_SAMPLES = 2
TRAIN_VAL_AR_HORIZONS = "1,10"
TRAIN_NUM_INFERENCE_STEPS = 8
TRAIN_CONTEXT_LEN = 4
TRAIN_MIXED_PRECISION = "bf16"
COUNTERFACTUAL_ENVS = "defender,jaws"
COUNTERFACTUAL_RULE = "multishot"
COUNTERFACTUAL_SAMPLES = 1

TRANSITIONS_ROOT = DRIVE_PROJECT_DIR / "data" / "transitions"
RUNS_ROOT = DRIVE_PROJECT_DIR / "runs"
CHECKPOINT_ROOT = DRIVE_PROJECT_DIR / "world_model" / "checkpoints" / "dit_encoded_rules_adv_colab"
PREVIEW_DIR = DRIVE_PROJECT_DIR / "previews"

print("Configured", len(GAME_STEMS), "GVGAI game stems")
print("Drive project dir:", DRIVE_PROJECT_DIR)

In [ ]:
# Mount Drive, clone this branch, and populate the linked GVGAI checkout.
from google.colab import drive
import os
import shutil
import subprocess
import sys

drive.mount("/content/drive")

def run(cmd, cwd=None, check=True):
    print("$", " ".join(str(c) for c in cmd))
    return subprocess.run([str(c) for c in cmd], cwd=str(cwd) if cwd else None, check=check)

if not (REPO_DIR / ".git").exists():
    run(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, REPO_DIR])
else:
    run(["git", "fetch", "origin", REPO_BRANCH], cwd=REPO_DIR)
    run(["git", "checkout", REPO_BRANCH], cwd=REPO_DIR)
    run(["git", "pull", "--ff-only", "origin", REPO_BRANCH], cwd=REPO_DIR)

GVGAI_DIR = REPO_DIR / "gvgai"
if not (GVGAI_DIR / ".git").exists():
    if GVGAI_DIR.exists():
        shutil.rmtree(GVGAI_DIR)
    run(["git", "clone", "--branch", GVGAI_BRANCH, REPO_URL, GVGAI_DIR])
else:
    run(["git", "fetch", "origin", "+refs/heads/*:refs/remotes/origin/*"], cwd=GVGAI_DIR)
run(["git", "checkout", "--detach", GVGAI_COMMIT], cwd=GVGAI_DIR)

for path in (TRANSITIONS_ROOT, RUNS_ROOT, CHECKPOINT_ROOT, PREVIEW_DIR):
    path.mkdir(parents=True, exist_ok=True)

os.chdir(REPO_DIR)
print("Repo:", REPO_DIR)
print("GVGAI:", GVGAI_DIR)

In [ ]:
# Install Python dependencies and compile GVGAI Java sources.
import subprocess
import sys
from pathlib import Path

os.chdir(REPO_DIR)
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])

sources_file = GVGAI_DIR / "sources_build.txt"
java_sources = sorted(str(p) for p in (GVGAI_DIR / "src").rglob("*.java"))
sources_file.write_text("\n".join(java_sources) + "\n")
(GVGAI_DIR / "out").mkdir(parents=True, exist_ok=True)
run([
    "javac", "--release", "8", "-encoding", "UTF-8",
    "-d", GVGAI_DIR / "out",
    "-cp", GVGAI_DIR / "gson-2.6.2.jar",
    "@" + str(sources_file),
], cwd=GVGAI_DIR)

GVGAI_CLASSPATH = f"{GVGAI_DIR / 'out'}:{GVGAI_DIR / 'gson-2.6.2.jar'}"
print("GVGAI classpath:", GVGAI_CLASSPATH)

In [ ]:
# Preflight checks before the overnight run.
import json
import numpy as np
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("bf16 supported:", torch.cuda.is_bf16_supported())

missing = [stem for stem in GAME_STEMS if not (GVGAI_DIR / "examples" / "data_collection" / f"{stem}.txt").is_file()]
if missing:
    raise FileNotFoundError(f"Missing GVGAI game stems: {missing}")

required_scripts = [
    REPO_DIR / "world_model" / "encode_transition.py",
    REPO_DIR / "world_model" / "train_dynamics.py",
    GVGAI_DIR / "src" / "tracks" / "singlePlayer" / "RunDataCollectionAgent.java",
]
for script in required_scripts:
    if not script.exists():
        raise FileNotFoundError(script)

for game in ["aliens", "chopper", "waves", "defender", "jaws"]:
    level = GVGAI_DIR / "examples" / "data_collection" / f"{game}_lvl0.txt"
    rows = [line.rstrip("\n") for line in level.read_text().splitlines() if line.strip()]
    widths = sorted({len(row) for row in rows})
    print(f"{game}: rows={len(rows)} widths={widths} native_pixels={max(widths) * 32}x{len(rows) * 32}")

print("Preflight complete.")

In [ ]:
# Collect RGB transitions into Drive-backed train/test splits.
import numpy as np
import subprocess

def shard_frame_count(env_dir: Path) -> int:
    total = 0
    for obs_path in sorted(env_dir.glob("shard_*/obs.npy")):
        total += int(np.load(obs_path, mmap_mode="r").shape[0])
    return total

def collect_stem(stem: str, split: str, target_frames: int, seed: int) -> None:
    split_root = TRANSITIONS_ROOT / split
    env_dir = split_root / stem
    existing = 0 if FORCE_RECOLLECT else shard_frame_count(env_dir)
    remaining = max(0, target_frames - existing)
    print(f"[{split}/{stem}] existing={existing:,} target={target_frames:,} remaining={remaining:,}")
    if remaining == 0:
        return
    cmd = [
        "java", "-cp", GVGAI_CLASSPATH,
        "tracks.singlePlayer.RunDataCollectionAgent",
        "--game", stem,
        "--agent", "tracks.singlePlayer.advanced.sampleMCTS.Agent",
        "--total-timesteps", str(remaining),
        "--num-envs", str(COLLECT_NUM_ENVS),
        "--output-root", str(split_root),
        "--chunk-size", str(COLLECT_CHUNK_SIZE),
        "--scale", str(COLLECT_SCALE),
        "--seed", str(seed),
    ]
    run(cmd, cwd=GVGAI_DIR)

for index, stem in enumerate(GAME_STEMS):
    collect_stem(stem, "train", TRAIN_FRAMES_PER_STEM, COLLECT_SEED + 1000 * index)
    collect_stem(stem, "test", TEST_FRAMES_PER_STEM, COLLECT_SEED + 1000 * index + 1)

print("Collection complete.")

In [ ]:
# Summarize raw transition shards before encoding.
import numpy as np

for split in ["train", "test"]:
    print(f"\n{split}")
    for stem in GAME_STEMS:
        env_dir = TRANSITIONS_ROOT / split / stem
        count = shard_frame_count(env_dir)
        first = next(env_dir.glob("shard_*/obs.npy"), None) if env_dir.exists() else None
        shape = tuple(np.load(first, mmap_mode="r").shape) if first else None
        print(f"  {stem:28s} frames={count:8,d} first_obs_shape={shape}")

In [ ]:
# Encode raw RGB transitions to WAN VAE latents.
os.chdir(REPO_DIR)
encode_cmd = [
    sys.executable, "-m", "world_model.encode_transition",
    "--transitions_root", str(TRANSITIONS_ROOT),
    "--encoded_subdir", "encoded",
    "--split", "both",
    "--down_scale", str(ENCODE_DOWN_SCALE),
    "--batch_size", str(ENCODE_BATCH_SIZE),
]
if FORCE_REENCODE:
    encode_cmd.append("--force")
run(encode_cmd, cwd=REPO_DIR)
print("Encoding complete.")

In [ ]:
# Start TensorBoard against the Drive-backed log directory.
from tensorboard import notebook

RUNS_ROOT.mkdir(parents=True, exist_ok=True)
notebook.start(f"--logdir {RUNS_ROOT} --port 6006")

In [ ]:
# Train the encoded dynamics model. Checkpoints and logs are written to Drive.
os.chdir(REPO_DIR)
train_cmd = [
    sys.executable, "-m", "world_model.train_dynamics",
    "--transitions_root", str(TRANSITIONS_ROOT),
    "--encoded_subdir", "encoded",
    "--num_actions", "7",
    "--context_len", str(TRAIN_CONTEXT_LEN),
    "--batch_size", str(TRAIN_BATCH_SIZE),
    "--num_workers", str(TRAIN_NUM_WORKERS),
    "--num_train_epochs", str(TRAIN_NUM_EPOCHS),
    "--max_train_steps", str(TRAIN_MAX_STEPS),
    "--lr", str(TRAIN_LR),
    "--mixed_precision", TRAIN_MIXED_PRECISION,
    "--num_inference_steps", str(TRAIN_NUM_INFERENCE_STEPS),
    "--validation_every", str(TRAIN_VALIDATION_EVERY),
    "--save_every", str(TRAIN_SAVE_EVERY),
    "--val_samples", str(TRAIN_VAL_SAMPLES),
    "--val_ar_horizons", TRAIN_VAL_AR_HORIZONS,
    "--checkpoint_dir", str(CHECKPOINT_ROOT),
    "--log_dir", str(RUNS_ROOT / "world_model_dynamics_adv"),
    "--counterfactual_envs", COUNTERFACTUAL_ENVS,
    "--counterfactual_rule", COUNTERFACTUAL_RULE,
    "--counterfactual_samples", str(COUNTERFACTUAL_SAMPLES),
    "--gradient_checkpointing",
]
run(train_cmd, cwd=REPO_DIR)
print("Training complete.")

In [ ]:
# Locate the latest checkpoint for post-training previews.
def latest_checkpoint(root: Path) -> Path:
    steps = sorted(
        [p for p in root.glob("step_*") if p.is_dir()],
        key=lambda p: int(p.name.split("_")[-1]),
    )
    if not steps:
        raise FileNotFoundError(f"No step_* checkpoints under {root}")
    return steps[-1]

FINAL_CKPT = latest_checkpoint(CHECKPOINT_ROOT)
print("Final checkpoint:", FINAL_CKPT)

In [ ]:
# WAN VAE reconstruction sanity checks: original RGB frame | decoded latent.
import matplotlib.pyplot as plt
import numpy as np
import torch
from world_model.dataset import obs_array_to_pixels
from world_model.model.net.vae import VAE

PREVIEW_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vae = VAE(checkpoint=None)
vae.freeze()
vae.to(device)

def tensor_to_hwc01(t: torch.Tensor) -> np.ndarray:
    return ((t.detach().float().cpu().clamp(-1, 1) + 1) * 0.5).permute(1, 2, 0).numpy()

for stem in GAME_STEMS:
    raw_shard = next((TRANSITIONS_ROOT / "test" / stem).glob("shard_*/obs.npy"), None)
    lat_shard = next((TRANSITIONS_ROOT / "encoded" / "test" / stem).glob("shard_*/latent.npy"), None)
    if raw_shard is None or lat_shard is None:
        print("skip", stem, "missing raw or encoded shard")
        continue
    raw = np.load(raw_shard, mmap_mode="r")
    lat = np.load(lat_shard, mmap_mode="r")
    row = min(5, raw.shape[0] - 1, lat.shape[0] - 1)
    original = obs_array_to_pixels(raw[row:row + 1])[0]
    z = torch.from_numpy(np.asarray(lat[row:row + 1], dtype=np.float32)).to(device=device, dtype=vae._dtype())
    with torch.no_grad():
        recon = vae.decode_latents(z)[0]
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(tensor_to_hwc01(original))
    axes[0].set_title("original")
    axes[1].imshow(tensor_to_hwc01(recon))
    axes[1].set_title("WAN VAE decode")
    for ax in axes:
        ax.axis("off")
    fig.suptitle(stem)
    out = PREVIEW_DIR / f"vae_reconstruction_{stem}.png"
    fig.tight_layout()
    fig.savefig(out, dpi=140)
    plt.show()
    print("saved", out)

In [ ]:
# Dynamics rollout previews: target frame | generated frame from the final checkpoint.
from functools import partial
import matplotlib.pyplot as plt
import torch
from world_model.dataset import MixedEncodedRolloutVideoDataset, encoded_dirs_with_rules, preprocess_latent
from world_model.inference import load_world_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
wm = load_world_model(FINAL_CKPT, num_actions=7, history_len=TRAIN_CONTEXT_LEN)
wm.eval()

def rule_label_from_stem(stem: str) -> str:
    if stem.endswith("_rules_multishot"):
        return "multishot"
    if stem.endswith("_rules_ricochet"):
        return "ricochet"
    return "normal"

for base in ["aliens", "chopper", "waves", "defender", "jaws"]:
    pairs = encoded_dirs_with_rules(TRANSITIONS_ROOT / "encoded" / "test", base)
    ds = MixedEncodedRolloutVideoDataset(
        pairs,
        seq_len=TRAIN_CONTEXT_LEN + 1,
        stride=1,
        num_actions=7,
    ).with_transform(partial(preprocess_latent, history_len=TRAIN_CONTEXT_LEN))
    indices = [0, max(0, len(ds) // 3), max(0, (2 * len(ds)) // 3)]
    for preview_idx, ds_idx in enumerate(indices):
        sample = ds[ds_idx]
        z_hist = sample["history_latents"].unsqueeze(0).to(device)
        hist_actions = sample["history_actions"].unsqueeze(0).to(device)
        rule_oh = sample["rule_onehot"].unsqueeze(0).to(device)
        transition_action = hist_actions[:, -1]
        target_z = sample["target_latent"].unsqueeze(0).to(device)
        with torch.no_grad():
            pred_z = wm.generate_next_frame(
                z_hist,
                hist_actions,
                transition_action,
                num_inference_steps=TRAIN_NUM_INFERENCE_STEPS,
                rule_onehot=rule_oh,
            )
            target_rgb = wm.decode_frames(target_z)[0]
            pred_rgb = wm.decode_video(pred_z)[0, 0]
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        axes[0].imshow(tensor_to_hwc01(target_rgb))
        axes[0].set_title("target")
        axes[1].imshow(tensor_to_hwc01(pred_rgb))
        axes[1].set_title("generated")
        for ax in axes:
            ax.axis("off")
        fig.suptitle(f"{base} preview {preview_idx}")
        out = PREVIEW_DIR / f"dynamics_rollout_{base}_{preview_idx}.png"
        fig.tight_layout()
        fig.savefig(out, dpi=140)
        plt.show()
        print("saved", out)